# Lab 2 optimizado: predicción de coeficientes de presión superficial TPU

**Objetivo:** verificar calidad, ejecutar EDA y comparar modelos para `cp_mean`, `cp_rms`, `cp_min` y `cp_max`.

El subconjunto contiene una geometría y siete direcciones. La validación reserva ángulos completos con `LeaveOneGroupOut`; no demuestra generalización a otras geometrías.


## 1. Configuración

In [ ]:
from pathlib import Path
import json, sys, warnings, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, max_error
from sklearn.inspection import permutation_importance
warnings.filterwarnings('ignore'); sns.set_theme(style='whitegrid')
RANDOM_STATE=42
SALIDA=Path('reports/resultados_notebook'); MODELOS_DIR=Path('models')
SALIDA.mkdir(parents=True,exist_ok=True); MODELOS_DIR.mkdir(parents=True,exist_ok=True)
print('Entorno preparado:',sys.version.split()[0],pd.__version__)

## 2. Carga y normalización del esquema

In [ ]:
CANDIDATOS=[Path('outputs/TPU_Gable_0-90_tabla_por_punto.csv'),Path('data/processed.parquet'),Path('data/processed.csv')]
ruta=next((p for p in CANDIDATOS if p.exists()),None)
if ruta is None: raise FileNotFoundError('No se encontró la tabla consolidada: '+', '.join(map(str,CANDIDATOS)))
df0=pd.read_parquet(ruta) if ruta.suffix=='.parquet' else pd.read_csv(ruta)
ALIASES={'cp_mean':['cp_mean','cp_mean_10min'],'cp_rms':['cp_rms','cp_rms_fluctuation_10min'],'cp_min':['cp_min','cp_min_0p2s_moving_average'],'cp_max':['cp_max','cp_max_0p2s_moving_average'],'theta':['theta','angle','angulo','wind_angle','wind_direction'],'sensor_id':['sensor_id','point_id','tap_id','sensor','point'],'surface':['surface','surface_id','face','face_id'],'x':['x','X','coord_x'],'y':['y','Y','coord_y'],'z':['z','Z','coord_z']}
low={str(c).lower():c for c in df0.columns}; ren={}
for canon,cands in ALIASES.items():
    for c in cands:
        if c.lower() in low: ren[low[c.lower()]]=canon; break
df=df0.rename(columns=ren).replace([np.inf,-np.inf],np.nan)
OBJ=['cp_mean','cp_rms','cp_min','cp_max']; faltan=[c for c in OBJ+['theta'] if c not in df]
if faltan: raise KeyError(f'Faltan columnas: {faltan}. Disponibles: {df.columns.tolist()}')
df['theta_sin']=np.sin(np.deg2rad(df.theta)); df['theta_cos']=np.cos(np.deg2rad(df.theta))
print(ruta,df.shape); display(df.head())

## 3. Calidad de datos y manejo de faltantes

La estrategia evita inventar valores de presión:

- Los objetivos faltantes y los ángulos faltantes no se imputan; esas filas se excluyen solo del ajuste correspondiente.
- Los predictores numéricos se imputan con la mediana calculada exclusivamente en el conjunto de entrenamiento de cada pliegue.
- Los predictores categóricos se imputan con la moda del conjunto de entrenamiento.
- Se agregan indicadores de ausencia para que el modelo pueda reconocer qué observaciones fueron imputadas.
- Los predictores con más de 40 % de datos faltantes, los completamente vacíos y los constantes se excluyen.
- Todo el procesamiento se realiza dentro de `Pipeline`, evitando fuga de información desde la dirección reservada.


In [ ]:
# Diagnóstico de datos faltantes
calidad = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "nulos": df.isna().sum(),
    "nulos_pct": 100 * df.isna().mean(),
    "unicos": df.nunique(dropna=True),
}).sort_values("nulos_pct", ascending=False)

display(calidad)
print("Duplicados de fila:", int(df.duplicated().sum()))
if {"sensor_id", "theta"} <= set(df.columns):
    print("Duplicados sensor-ángulo:", int(df.duplicated(["sensor_id", "theta"]).sum()))
print("Ángulos:", sorted(df["theta"].dropna().unique()))

# Política conservadora de faltantes
UMBRAL_NULOS_PREDICTOR = 40.0  # porcentaje máximo admisible por predictor
columnas_totalmente_vacias = calidad.index[calidad["nulos_pct"] == 100].tolist()
if columnas_totalmente_vacias:
    print("Columnas completamente vacías, se excluirán:", columnas_totalmente_vacias)

# No se imputan objetivos ni el ángulo usado como grupo de validación.
# Esas filas se excluyen dentro de cada objetivo para no inventar valores físicos de Cp.
for objetivo in OBJ:
    n = int(df[objetivo].isna().sum())
    if n:
        print(f"{objetivo}: se excluirán {n} filas sin valor observado durante su modelado.")

filas_sin_theta = int(df["theta"].isna().sum())
if filas_sin_theta:
    print(f"theta: se excluirán {filas_sin_theta} filas porque no pueden asignarse a un pliegue angular.")

assert df["theta"].nunique(dropna=True) >= 2, "Se requieren al menos dos ángulos no nulos para validar."


In [ ]:
nulos=(100*df.isna().mean()).sort_values(ascending=False)
plt.figure(figsize=(10,4)); sns.barplot(x=nulos.index,y=nulos.values,color='#3498db'); plt.xticks(rotation=60); plt.ylabel('Nulos (%)'); plt.tight_layout(); plt.savefig(SALIDA/'eda_nulos.png',dpi=180); plt.show()

## 4. EDA, distribuciones y correlación

In [ ]:
display(df[OBJ].describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]).T)
fig,axs=plt.subplots(2,2,figsize=(13,9))
for ax,t in zip(axs.ravel(),OBJ): sns.histplot(df[t].dropna(), bins=35, kde=True, ax=ax, color='#3498db'); ax.set_title(t)
plt.tight_layout(); plt.savefig(SALIDA/'eda_distribuciones.png',dpi=180); plt.show()

In [ ]:
corr=df.select_dtypes(include=np.number).corr(method='spearman')
plt.figure(figsize=(11,8)); sns.heatmap(corr,cmap='vlag',center=0); plt.title('Correlación de Spearman'); plt.tight_layout(); plt.savefig(SALIDA/'eda_correlacion.png',dpi=180); plt.show()
display(corr.loc[OBJ].round(3))

In [ ]:
g=df.groupby('theta')[OBJ].agg(['mean','std']); fig,axs=plt.subplots(2,2,figsize=(13,9),sharex=True)
for ax,t in zip(axs.ravel(),OBJ): ax.errorbar(g.index,g[(t,'mean')],yerr=g[(t,'std')],marker='o',capsize=3); ax.set(title=t,xlabel='Ángulo (°)')
plt.tight_layout(); plt.savefig(SALIDA/'eda_por_angulo.png',dpi=180); plt.show()

## 5. Predictores

Se excluyen H, B, D y H/B si son constantes. `sensor_id` tampoco se usa como etiqueta memorística. Se emplean coordenadas, superficie y codificación circular del ángulo.

In [ ]:
candidatas = ["x", "y", "z", "surface", "theta_sin", "theta_cos"]
FEATURES = [
    c for c in candidatas
    if c in df.columns
    and df[c].nunique(dropna=True) > 1
    and 100 * df[c].isna().mean() <= UMBRAL_NULOS_PREDICTOR
]
excluidas = [c for c in candidatas if c in df.columns and c not in FEATURES]
print("Predictores excluidos por ser constantes o superar el umbral de nulos:", excluidas)
if not FEATURES:
    raise ValueError("No hay predictores variables con datos suficientes; revise el esquema y los faltantes.")

NUM = [c for c in FEATURES if pd.api.types.is_numeric_dtype(df[c]) and c != "surface"]
CAT = [c for c in FEATURES if c not in NUM]
transformadores = []
if NUM:
    transformadores.append((
        "num",
        Pipeline([
            ("imputacion", SimpleImputer(strategy="median", add_indicator=True)),
            ("escala", StandardScaler()),
        ]),
        NUM,
    ))
if CAT:
    try:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)
    transformadores.append((
        "cat",
        Pipeline([
            ("imputacion", SimpleImputer(strategy="most_frequent", add_indicator=True)),
            ("onehot", encoder),
        ]),
        CAT,
    ))
pre = ColumnTransformer(transformadores, remainder="drop")
print("Predictores utilizados:", FEATURES)
print("Numéricos: imputación por mediana + indicador de ausencia:", NUM)
print("Categóricos: imputación por moda + indicador + one-hot:", CAT)


## 6. Comparación de algoritmos con validación angular

In [ ]:
ALG={'Dummy':DummyRegressor(strategy='median'),'Ridge':Ridge(alpha=1.0),'RandomForest':RandomForestRegressor(n_estimators=350,min_samples_leaf=2,random_state=42,n_jobs=-1),'ExtraTrees':ExtraTreesRegressor(n_estimators=350,min_samples_leaf=2,random_state=42,n_jobs=-1),'HistGradientBoosting':HistGradientBoostingRegressor(max_iter=250,learning_rate=.05,l2_regularization=1,random_state=42)}
def mets(y,p): return {'MAE':mean_absolute_error(y,p),'RMSE':mean_squared_error(y,p)**.5,'R2':r2_score(y,p),'Error_max':max_error(y,p),'Sesgo':float(np.mean(p-y))}
logo=LeaveOneGroupOut(); filas=[]; preds=[]
for target in OBJ:
 d=df[FEATURES+[target,'theta']].dropna(subset=[target,'theta']); X,y,grp=d[FEATURES],d[target],d.theta
 for nombre,est in ALG.items():
  for fold,(tr,te) in enumerate(logo.split(X,y,grp),1):
   pipe=Pipeline([('pre',clone(pre)),('modelo',clone(est))]); pipe.fit(X.iloc[tr],y.iloc[tr]); p=pipe.predict(X.iloc[te])
   filas.append({'objetivo':target,'modelo':nombre,'fold':fold,'angulo_prueba':grp.iloc[te].iloc[0],**mets(y.iloc[te],p)})
   preds.append(pd.DataFrame({'objetivo':target,'modelo':nombre,'angulo':grp.iloc[te].to_numpy(),'observado':y.iloc[te].to_numpy(),'predicho':p,'residuo':y.iloc[te].to_numpy()-p}))
metricas_fold=pd.DataFrame(filas); pred_cv=pd.concat(preds,ignore_index=True)
resumen=metricas_fold.groupby(['objetivo','modelo']).agg(MAE=('MAE','mean'),MAE_std=('MAE','std'),RMSE=('RMSE','mean'),RMSE_std=('RMSE','std'),R2=('R2','mean'),R2_std=('R2','std'),Error_max=('Error_max','max'),Sesgo=('Sesgo','mean')).reset_index()
display(resumen.sort_values(['objetivo','MAE']).round(4))

## 7. Selección y diagnósticos

In [ ]:
mejores=resumen[resumen.modelo!='Dummy'].sort_values(['objetivo','MAE']).groupby('objetivo',as_index=False).first()
base=resumen[resumen.modelo=='Dummy'][['objetivo','MAE','RMSE']].rename(columns={'MAE':'MAE_base','RMSE':'RMSE_base'})
comp=mejores.merge(base,on='objetivo'); comp['mejora_MAE_pct']=100*(1-comp.MAE/comp.MAE_base); comp['mejora_RMSE_pct']=100*(1-comp.RMSE/comp.RMSE_base)
display(comp.round(4))

In [ ]:
fig,axs=plt.subplots(2,2,figsize=(13,10))
for ax,t in zip(axs.ravel(),OBJ):
 m=mejores.loc[mejores.objetivo==t,'modelo'].iloc[0]; q=pred_cv[(pred_cv.objetivo==t)&(pred_cv.modelo==m)]; ax.scatter(q.observado,q.predicho,s=18,alpha=.55); lo=min(q.observado.min(),q.predicho.min()); hi=max(q.observado.max(),q.predicho.max()); ax.plot([lo,hi],[lo,hi],'r--'); ax.set(title=f'{t}: {m}',xlabel='Observado',ylabel='Predicho')
plt.tight_layout(); plt.savefig(SALIDA/'predicho_observado.png',dpi=180); plt.show()

## 8. Importancia por permutación y modelado final

In [ ]:
imps=[]
for target in OBJ:
 nombre=mejores.loc[mejores.objetivo==target,'modelo'].iloc[0]; d=df[FEATURES+[target]].dropna(subset=[target]); pipe=Pipeline([('pre',clone(pre)),('modelo',clone(ALG[nombre]))]).fit(d[FEATURES],d[target]); joblib.dump(pipe,MODELOS_DIR/f'mejor_modelo_{target}.joblib')
 r=permutation_importance(pipe,d[FEATURES],d[target],scoring='neg_mean_absolute_error',n_repeats=20,random_state=42,n_jobs=-1)
 imps.append(pd.DataFrame({'objetivo':target,'modelo':nombre,'variable':FEATURES,'importancia_media':r.importances_mean,'importancia_std':r.importances_std}))
importancia=pd.concat(imps,ignore_index=True); display(importancia.sort_values(['objetivo','importancia_media'],ascending=[True,False]).round(4))

## 9. Exportación

In [ ]:
calidad.to_csv(SALIDA/'reporte_datos_faltantes.csv')
metricas_fold.to_csv(SALIDA/'metricas_por_pliegue.csv',index=False); resumen.to_csv(SALIDA/'resumen_modelos.csv',index=False); comp.to_csv(SALIDA/'comparacion_linea_base.csv',index=False); pred_cv.to_csv(SALIDA/'predicciones_validacion_angular.csv',index=False); importancia.to_csv(SALIDA/'importancia_permutacion.csv',index=False)
config={'semilla':42,'validacion':'LeaveOneGroupOut por theta','predictores':FEATURES,'objetivos':OBJ,'modelos':list(ALG)}; (SALIDA/'configuracion.json').write_text(json.dumps(config,indent=2,ensure_ascii=False),encoding='utf-8'); print('Resultados guardados en',SALIDA)

## 10. Límites de validez

- La evaluación mide generalización angular dentro de una sola geometría.
- No permite afirmar desempeño para H/B, dimensiones o edificios no observados.
- Los extremos corresponden al registro experimental disponible, no a un periodo de retorno universal.
- Antes de aplicar resultados al diseño estructural, deben contrastarse con los datos originales TPU y con la norma de viento aplicable.